In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("../data/cleaned/news_1.csv")

In [49]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix

X = df["text"]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y ,random_state=42)

vectorizer = CountVectorizer(stop_words="english", max_features=50000)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [50]:
tfidf_vectorizer = TfidfVectorizer(stop_words="english", max_features=50000)
tfidf_X_train_vec = tfidf_vectorizer.fit_transform(X_train)
tfidf_X_test_vec = tfidf_vectorizer.transform(X_test)

In [52]:
NB_tfidf = MultinomialNB()

NB_tfidf.fit(tfidf_X_train_vec, y_train)
y_pred_tfidf = NB_tfidf.predict(tfidf_X_test_vec)

In [53]:
accuracy_score(y_test, y_pred_tfidf)

0.9375278396436526

In [54]:
confusion_matrix(y_test, y_pred_tfidf)

array([[4398,  298],
       [ 263, 4021]])

In [38]:
len(vectorizer.vocabulary_)

50000

In [46]:
NB = MultinomialNB()

NB.fit(X_train_vec, y_train)
y_pred = NB.predict(X_test_vec)

In [47]:
accuracy_score(y_test, y_pred)

0.9513363028953229

In [48]:
confusion_matrix(y_test, y_pred)

array([[4459,  237],
       [ 200, 4084]])

In [29]:
from IPython.display import display

pd.set_option('display.max_rows', 10)
df_misclassified = pd.DataFrame({
    "text": X_test.values,
    "Actual": y_test.values,
    "Prediction": y_pred
})

df_misclassified = df_misclassified[df_misclassified["Actual"] != df_misclassified["Prediction"]]

styled = df_misclassified.head(10).style.set_table_styles(
    [{"selector": "tbody tr", "props": [("height", "20em")]}]
)
display(styled)

In [30]:
# subject vs label
print(df.groupby("subject")["target"].agg(["count","mean"]).sort_values("count", ascending=False).head(20))

# check if 'Reuters' appears systematically in texts and how it correlates
df["has_reuters"] = df["text"].str.contains(r"Reuters", case=False, na=False)
print(df.groupby("has_reuters")["target"].agg(["count","mean"]))

                 count  mean
subject                     
politicsNews     11272   1.0
worldnews        10145   1.0
News              9050   0.0
politics          6841   0.0
left-news         4459   0.0
Government News   1570   0.0
US_News            783   0.0
Middle-east        778   0.0
             count      mean
has_reuters                 
False        23198  0.001681
True         21700  0.985161


In [31]:
keywords = ["trump", "donald", "obama", "biden", "china", "putin"]

for word in keywords:
    mask = df["text"].str.contains(word, case=False, na=False)
    print(word, "→", df.loc[mask, "target"].mean(), "| count:", mask.sum())

trump → 0.4324821612888749 | count: 22283
donald → 0.5008321254093521 | count: 18627
obama → 0.38736212376145074 | count: 10698
biden → 0.31151832460732987 | count: 382
china → 0.8005833029529712 | count: 2743
putin → 0.5373134328358209 | count: 1742


In [32]:
import numpy as np

feature_names = vectorizer.get_feature_names_out()

log_probs = NB.feature_log_prob_

# top words for class 0
top_fake = np.argsort(log_probs[0])[-20:]

# top words for class 1
top_real = np.argsort(log_probs[1])[-20:]

print("Top words for FAKE:")
print([feature_names[i] for i in top_fake])

print("\nTop words for REAL:")
print([feature_names[i] for i in top_real])


Top words for FAKE:
['house', 'america', 'american', 'twitter', 'media', 'white', 'state', 'time', 'hillary', 'new', 'news', 'donald', 'like', 'obama', 'clinton', 'just', 'people', 'president', 'said', 'trump']

Top words for REAL:
['security', 'donald', 'campaign', 'election', 'party', 'washington', 'told', 'year', 'people', 'united', 'republican', 'house', 'states', 'new', 'government', 'state', 'president', 'reuters', 'trump', 'said']
